<a href="https://colab.research.google.com/github/jdmartinev/ST1613-AppliedML-/blob/main/Semana02/Preprocesamiento_de_datos_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Semana 2 - Preprocesamiento de datos**

## **Objetivos de aprendizaje**

- Reconocer el impacto de la calidad y cantidad de datos en los algoritmos de ML
- Explicar un proceso t?pico de recolecci?n y transformaci?n de datos en el flujo de trabajo de ML
- Identificar cuando implementar transformaciones de caracter?sticas como imputaci?n, escalamiento, one-hot-encoding

## **Ejemplo 1: Recomendaci?n de canciones usando $k$-NN**

En este ejemplo usaremos $k-$NN para predecir si a un usuario le gustar? una canci?n o no.

Intuici?n: 

- Encuentre las canciones similares en el conjunto de entrenamiento a la canci?n a la que queremos hacer la predicci?n 
- Por mayor?a, escoger si al usuario le gustaron o no estas canciones
- Asignar la etiqueta a la nueva canci?n seg?n el voto mayoritario

![Spotify recommendation](https://drive.google.com/uc?id=18lBxaBGX3n8BPxwyOnybK2CqtMHDp9Up)



## **Descarga de datos (opci?n 1)**

Lo primero que har? ser? descargar los datos con los que va a trabajar:

- Tabla de datos de clientes (archivo csv).

Este archivo quedar? en la ruta _/content/data.csv asociada con el sistema operativo del servidor en el que se est? ejecutando google colab.

In [ ]:
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1UAeE2IQvuFxvwep1ThK7_DnpBfUB7eyr' -O data_spotify.csv



## **Descarga de datos (opci?n 2)**

Vincular google colab con google drive

https://www.youtube.com/watch?v=6UnCrulz-fE



## **Importar librer?as necesarias**

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import HTML

sys.path.append("code/.")

#import mglearn
from IPython.display import display
#from plotting_functions import *

# Classifiers and regressors
from sklearn.dummy import DummyClassifier, DummyRegressor

# Preprocessing and pipeline
from sklearn.impute import SimpleImputer

# train test split and cross validation
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
#from utils import *

pd.set_option("display.max_colwidth", 200)

## **Entrenamiento del modelo**


In [ ]:
# Cargar los datos
spotify_df = pd.read_csv("data_spotify.csv", index_col=0)
spotify_df.head()

In [ ]:
#Particiones de entrenamiento, validaci?n y prueba
train_df, test_df = train_test_split(spotify_df, test_size=0.20, random_state=123)
train_df, val_df = train_test_split(train_df, test_size = 0.20, random_state=123)


In [ ]:
#Definir variable objetivo y eliminar variables no num?ricas
X_train, y_train = (
    train_df.drop(columns=["song_title", "artist", "target"]),
    train_df["target"],
)

X_val, y_val = (
    val_df.drop(columns=["song_title", "artist", "target"]),
    val_df["target"],
)

X_test, y_test = (
    test_df.drop(columns=["song_title", "artist", "target"]),
    test_df["target"],
)


In [ ]:
# Crear un clasificador KNN con diferentes n?meros de vecinos
n_neighbors = [1,3,5,7,9,11,13,15]
accuracy = []

for k in n_neighbors:
  clf = Pipeline(steps=[("classifier", KNeighborsClassifier(n_neighbors=k))])

  # Entrenar el clasificador con los datos de entrenamiento
  clf.fit(X_train, y_train)

  # Evaluar el rendimiento del clasificador en los datos de validaci?n
  accuracy.append(clf.score(X_val, y_val))

In [ ]:
gfg = sns.barplot(x = n_neighbors,y = accuracy)
gfg.set(xlabel ="Number of neighbors", ylabel = "Accuracy (%)", title ='k-NN Classification')

In [ ]:
#Re-entrenar el modelo con los datos de entrenamiento y validaci?n para el valor de k ?ptimo
k_opt = n_neighbors[np.argmax(accuracy)]
print(f"El n?mero ?ptimo de vecinos es {k_opt}")
X_train_test = pd.concat([X_train,X_val],axis=0)
y_train_test = np.hstack([y_train,y_val])

clf = Pipeline(steps=[("classifier", KNeighborsClassifier(n_neighbors=k_opt))])
clf.fit(X_train_test, y_train_test)
#Validar el rendimiento del clasificador con los datos de prueba
accuracy = clf.score(X_test, y_test)

print(f"La tasa de acierto de clasificaci?n en el conjunto de test es {accuracy:0.2f}")

## ?C?mo se calculan los vecinos m?s cercanos?

Notaci?n matem?tica:

Cada fila de un dataframe ser? para nosotros un vector de n posiciones (vector de caracter?sticas):

$\mathbf{x}^{(i)} = \begin{bmatrix}
x_1^{(i)} \\
x_2^{(i)} \\
\vdots \\
x_n^{(i)}
\end{bmatrix} \in \Re^{n}$

El super-?ndice (i) denota la fila del dataframe

La distancia euclidiana sirve para saber qu? tan lejos o cerca est?n dos vectores de caracter?sticas. Se calcula de la siguiente forma: 

$d(\mathbf{x}^{(1)},\mathbf{x}^{(2)}) = \sqrt{(x_1^{(1)} - x_1^{(2)})^2 + (x_2^{(1)} - x_2^{(2)})^2 + ... + (x_n^{(1)} - x_n^{(2)})^2}$

In [ ]:
#Tomemos dos filas del dataframe y calculemos la distancia euclidiana
two_songs = X_train.sample(2, random_state=42)
two_songs

In [ ]:
from scipy.spatial.distance import pdist, squareform

In [ ]:
dist_matrix = squareform(pdist(two_songs, metric='euclidean'))
print(dist_matrix)

In [ ]:
#Consideremos ahora ?nicamente dos caracter?sticas (distancia y tiempo)
two_songs_subset = two_songs[["duration_ms", "tempo"]]
two_songs_subset

In [ ]:
dist_matrix = squareform(pdist(two_songs_subset, metric='euclidean'))
print(dist_matrix)

?Encontramos alg?n problema?

- La distancia est? completamente dominada por las caracter?sticas con valores m?s grandes.
- Las caracter?sticas con valores m?s peque?os est?n siendo ignoradas.
- ?Importa?
  - ?S?! La escala se basa en c?mo se recopilaron los datos.
  - Las caracter?sticas en una escala m?s peque?a pueden ser altamente informativas y no hay una buena raz?n para ignorarlas.
  - Queremos que nuestro modelo sea robusto y no sea sensible a la escala.

Este problema aplica para todos los modelos que usan distancias en su formulaci?n, regresi?n lineal, log?stica, redes neuronales.

?No aplica para modelos basados en ?rboles de decisi?n!

### Escalamiento usando [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) de `scikit-learn`

- Utilizaremos [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) de `scikit-learn`, que es un `transformer`.   
- Por ahora nos centraremos en la sintaxis. 

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()  # Crear objeto transformador de caracter?sticas
scaler.fit(X_train)  # ajustar el transformador en la divisi?n de entrenamiento
X_train_scaled = scaler.transform(X_train)  # transformar la divisi?n de entrenamiento
X_val_scaled = scaler.transform(X_val)   # transformar la divisi?n de validaci?n
X_test_scaled = scaler.transform(X_test)  # transformar la divisi?n de prueba

In [ ]:
X_train

Examinemos el valor transformado de la caracter?stica `energ?a` en la primera fila.

In [ ]:
X_train['energy'].iloc[0] 

In [ ]:
(X_train['energy'].iloc[0] - np.mean(X_train['energy']))/ X_train['energy'].std()


In [ ]:
pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index).head().round(3)

## Paradigma `fit` y `transform` para transformadores

- `sklearn` utiliza los paradigmas `fit` y `transform` para transformaciones de caracter?sticas.
- Ajustamos el transformador en la divisi?n de entrenamiento y luego transformamos tanto la divisi?n de entrenamiento como la de prueba.
- Aplicamos las mismas transformaciones en las divisiones de validacion y prueba.

## Resumen de la API `sklearn`: transformadores

Supongamos que transformer es un transformador utilizado para cambiar la representaci?n de entrada, por ejemplo, para completar valores faltantes o escalar caracter?sticas num?ricas.

```
transformer.fit(X_train, [y_train])
X_train_transformed = transformer.transform(X_train)
X_test_transformed = transformer.transform(X_test)
```

- Puedes pasar y_train en fit pero generalmente se ignora. Te permite pasarlo solo para ser coherente con el uso habitual del m?todo fit de sklearn.
- Tambi?n puedes llevar a cabo el ajuste y la transformaci?n en una sola llamada usando `fit_transform`. Pero ten en cuenta que debes usarlo solo en la divisi?n de entrenamiento y no en la divisi?n de validaci?n o de prueba.

## Re-entrenamiento del modelo incluyendo el escalamiento



In [ ]:
# Crear un clasificador KNN con diferentes n?meros de vecinos
n_neighbors = [1,3,5,7,9,11,13,15]
accuracy = []

for k in n_neighbors:
  clf = Pipeline(steps=[("classifier", KNeighborsClassifier(n_neighbors=k))])

  # Entrenar el clasificador con los datos de entrenamiento
  clf.fit(X_train_scaled, y_train)

  # Evaluar el rendimiento del clasificador en los datos de validaci?n
  accuracy.append(clf.score(X_val_scaled, y_val))

In [ ]:
#Re-entrenar el modelo con los datos de entrenamiento y validaci?n para el valor de k ?ptimo
k_opt = n_neighbors[np.argmax(accuracy)]
print(f"El n?mero ?ptimo de vecinos es {k_opt}")
X_train_test_scaled = np.vstack([X_train_scaled,X_val_scaled])
y_train_test = np.hstack([y_train,y_val])

clf = Pipeline(steps=[("classifier", KNeighborsClassifier(n_neighbors=k_opt))])
clf.fit(X_train_test_scaled, y_train_test)
#Validar el rendimiento del clasificador con los datos de prueba
accuracy = clf.score(X_test_scaled, y_test)

print(f"La tasa de acierto de clasificaci?n en el conjunto de test es {accuracy:0.2f}")

La tasa de acierto es mejor (casi 10%) con los datos escalados

### Utilizando `Pipelines` de `sklearn`

Un pipeline en scikit-learn es un objeto que permite concatenar varios pasos de preprocesamiento y modelado en un solo estimador. Los pipelines son ?tiles para automatizar el flujo de trabajo, garantizar que los pasos se apliquen consistentemente en los conjuntos de entrenamiento y prueba, y facilitar la integraci?n con la validaci?n cruzada y la optimizaci?n de hiperpar?metros.

In [ ]:
# Crear un clasificador KNN con diferentes n?meros de vecinos
n_neighbors = [1,3,5,7,9,11,13,15]
accuracy = []

for k in n_neighbors:
  clf = Pipeline(steps=[("preprocessor", StandardScaler()), ("classifier", KNeighborsClassifier(n_neighbors=k))])

  # Entrenar el clasificador con los datos de entrenamiento
  clf.fit(X_train, y_train)

  # Evaluar el rendimiento del clasificador en los datos de validaci?n
  accuracy.append(clf.score(X_val, y_val))

In [ ]:
gfg = sns.barplot(x = n_neighbors,y = accuracy)
gfg.set(xlabel ="Number of neighbors", ylabel = "Accuracy (%)", title ='k-NN Classification')

In [ ]:
#Re-entrenar el modelo con los datos de entrenamiento y validaci?n para el valor de k ?ptimo
k_opt = n_neighbors[np.argmax(accuracy)]
print(f"El n?mero ?ptimo de vecinos es {k_opt}")
X_train_test = pd.concat([X_train,X_val],axis=0)
y_train_test = np.hstack([y_train,y_val])

clf = Pipeline(steps=[("preprocessor", StandardScaler()),("classifier", KNeighborsClassifier(n_neighbors=k_opt))])
clf.fit(X_train_test, y_train_test)
#Validar el rendimiento del clasificador con los datos de prueba
accuracy = clf.score(X_test, y_test)

print(f"La tasa de acierto de clasificaci?n en el conjunto de test es {accuracy:0.2f}")